# 03 - Train, evaluate and compare

**Settings:** Accelerator = GPU T4 x2 (one GPU is used; T4 is recommended over P100), Internet = On.
**Inputs:** add the outputs of notebooks 01 and 02 (*Add Input > Your Work*). To continue in a
new session, also add this notebook's previous output: finished runs are copied and skipped.

Order: baselines B1-B4, then the U-Net experiments, then the comparison report.
Each U-Net run takes roughly 45-60 min on a T4, so all seven fit in one 12-hour session;
if not, remove some names from `EXPERIMENTS` and run the rest in a second session.

In [ ]:
# ---- Settings -------------------------------------------------------------
GITHUB_REPO = "https://github.com/mhdnasim6-boop/urban-flood-mapping.git"   # <- edit
BRANCH = "main"
# ---------------------------------------------------------------------------
# Private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets).
# Optional: a secret named HF_TOKEN (Hugging Face read token) avoids download rate limits.
import os, subprocess
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
url = GITHUB_REPO
if secret("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
subprocess.run(["rm", "-rf", "/tmp/ufm"])
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/tmp/ufm"], check=True)
os.chdir("/tmp/ufm")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
EXPERIMENTS = ["D_full", "B5_authors_baseline", "A_raw", "B_raw_phys", "C_raw_aux",
               "E_no_coherence", "F_no_vh"]   # proposed model first, then comparison/ablation
RUN_BASELINES = True
EXTRA = []            # config overrides for every run, e.g. ["train.epochs=30"]

import glob, shutil
from pathlib import Path
TRAINVAL = glob.glob("/kaggle/input/**/ufm-trainval", recursive=True)[0]
TEST = glob.glob("/kaggle/input/**/ufm-test", recursive=True)[0]
RUNS = Path("/kaggle/working/runs")
RUNS.mkdir(parents=True, exist_ok=True)
for prev in glob.glob("/kaggle/input/**/runs", recursive=True):   # resume from earlier sessions
    shutil.copytree(prev, RUNS, dirs_exist_ok=True)
print("train/val:", TRAINVAL, "\ntest:", TEST, "\nfinished runs:", sorted(p.parent.parent.name for p in RUNS.glob("*/test/test_metrics.csv")))

In [ ]:
!pip install -q segmentation-models-pytorch tabulate rasterio 2>/dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
if RUN_BASELINES and not (RUNS / "baselines/B4/test/test_metrics.csv").exists():
    !python scripts/run_baselines.py --test-root {TEST} --trainval-root {TRAINVAL} --out {RUNS}/baselines

In [ ]:
import subprocess
for exp in EXPERIMENTS:
    if (RUNS / exp / "test" / "test_metrics.csv").exists():
        print(f"skip {exp}: already done"); continue
    subprocess.run(["python", "scripts/train.py", "--config", f"configs/experiments/{exp}.yaml",
                    "--data-root", TRAINVAL, "--out", str(RUNS), *EXTRA], check=True)
    subprocess.run(["python", "scripts/evaluate.py", "--run", str(RUNS / exp), "--test-root", TEST, "--mc", "0"], check=True)
    if exp == "D_full":   # uncertainty maps + MC-averaged accuracy for the proposed model
        subprocess.run(["python", "scripts/evaluate.py", "--run", str(RUNS / exp), "--test-root", TEST,
                        "--mc", "20", "--tag", "mc"], check=True)

In [ ]:
!python scripts/summarize.py --runs {RUNS} --test-root {TEST} --out /kaggle/working/report

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("/kaggle/working/report/comparison.md").read()))
display(Image("/kaggle/working/report/f1_bars.png"))
for p in sorted(glob.glob("/kaggle/working/report/maps_*.png")):
    display(Image(p))